In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import torch
import psutil
import sys
import os
from matplotlib.ticker import ScalarFormatter
import yaml

In [2]:
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.default'] = 'rm'

plt.rc("font", family="serif", size=30)
plt.rc("axes", titlesize="medium")

plt.rcParams['xtick.labelsize'] = 30
plt.rcParams['ytick.labelsize'] = 30

plt.rcParams["axes.formatter.limits"] = [-3,3]

rect_double    = (0.05, 0.12, 0.98, 0.97) # left, bottom, right, top
rect_double_with_legend = (0.14, 0.12, 0.98, 0.97) # left, bottom, right, top

In [3]:
cd /afs/cern.ch/user/f/fernst/PHD/01_Machine_Learning/CaloINN/src

/eos/home-f/fernst/ML/CaloINN/src


/afs/cern.ch/user/f/fernst/eos/ML/Conda/envs/CaloINN/lib/python3.12/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
from documenter import Documenter
from trainer import ECAETrainer
import data_util
import plotting

In [5]:
print(psutil.virtual_memory())
torch.set_default_dtype(torch.float32)

svmem(total=30521311232, available=15617060864, percent=48.8, used=14217207808, free=12901142528, active=5588238336, inactive=9598377984, buffers=4820992, cached=3398139904, shared=6455296, slab=1223860224)


In [6]:
def load_trainer(directory, use_cuda=True):
    use_cuda = torch.cuda.is_available() and use_cuda
    device = 'cuda:0' if use_cuda else 'cpu' 


    with open(os.path.join(directory, r"params.yaml")) as f:
        params = yaml.load(f, Loader=yaml.FullLoader)
        
    doc = Documenter(params['run_name'], existing_run=True, basedir=directory,
                    log_name="log_jupyter.txt", read_only=True)
        
    trainer = ECAETrainer(params, device, doc)
    
    return trainer, params, device, doc

In [7]:
trainer, params, device, doc = load_trainer(r"/afs/cern.ch/user/f/fernst/PHD/01_Machine_Learning/CaloINN/results/2024_05_29_110611_INN_photons_eta_20_correct_noise")
del trainer.train_loader
del trainer.test_loader
trainer.load()

Using the directory: /afs/cern.ch/user/f/fernst/PHD/01_Machine_Learning/CaloINN/results/2024_05_29_110611_INN_photons_eta_20_correct_noise
Using the directory: /afs/cern.ch/user/f/fernst/PHD/01_Machine_Learning/CaloINN/results/2024_05_29_110611_INN_photons_eta_20_correct_noise/VAE
Device:  cpu
Using layers tensor([ 0,  1,  2,  3, 12], dtype=torch.int32)
Fixed 1 negative layers
Removed 11 of 127271 events (0.01%)
Adding noise to the INN loaders
Input dimension: 6246
num samples out of bounds: 4207
log transformation is used
number of trainable parameters: 259487672
number of parameters: 259681299
CINN(
  (model): GraphINN(
    (module_list): ModuleList(
      (0): LogTransformation()
      (1): FixedAffineTransform()
      (2): CubicSplineBlock(
        (softplus): Softplus(beta=0.5, threshold=20.0)
        (subnet): Subnet(
          (layers): Sequential(
            (0): Linear(in_features=3124, out_features=256, bias=True)
            (1): SiLU()
            (2): Linear(in_features=2

In [8]:
save_dir = "/afs/cern.ch/user/f/fernst/FrozenShowerInputSamples/eta_020_INN_samples"
old_file = trainer.params["data_path"]

energies = []
for i in range(8, 23):
    energies.append(2**i)

for einc in energies:
    x, c = trainer.generate(5000, einc=einc, batch_size=1000)
    x, c, layer_boundaries = data_util.postprocess(x, c, trainer.layer_boundaries, trainer.negative_layers)
    
    new_file = os.path.join(save_dir, f"energy_{int(einc)}", "dataset_combined.hdf5")
    os.makedirs(os.path.dirname(new_file), exist_ok=True)
    print(new_file)
    data_util.save_data(new_file, old_file, x, c, layer_boundaries)


/afs/cern.ch/user/f/fernst/FrozenShowerInputSamples/eta_020_INN_samples/energy_256/dataset_combined.hdf5
/afs/cern.ch/user/f/fernst/FrozenShowerInputSamples/eta_020_INN_samples/energy_512/dataset_combined.hdf5
/afs/cern.ch/user/f/fernst/FrozenShowerInputSamples/eta_020_INN_samples/energy_1024/dataset_combined.hdf5


KeyboardInterrupt: 